### Mesh convergence study

In [ ]:
import os
import sys
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

# Load these modules here so that the petsc4py.init() call can handle the CLI args.
import dolfin as dl

sys.path.append(os.environ.get("HIPPYLIB_PATH"))
import hippylib as hp

sys.path.append(os.path.join(os.getenv("DT4CO_PATH"), "src"))
from dt4co.utils.mesh_utils import load_mesh

import nibabel

In [ ]:
def l2_norm(u):
    """Compute the L2 norm of a function."""
    norm = dl.assemble(u**2 * dl.dx)
    return np.sqrt(norm)

def l2_error(u, u_ref):
    """Compute the L2 error between a function and a reference function."""
    error = dl.assemble((u - u_ref)**2 * dl.dx)
    return np.sqrt(error)

In [ ]:
# Set LaTeX rendering for all text
mpl.rcParams['text.usetex'] = True
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.serif'] = ['Computer Modern']
mpl.rcParams['font.size'] = 16

gray = "#191919"
blue = "#305CDE"
red = "#EE2400"
utorange = "#BF5700"
utblue = "#00A9B7"

Read in mesh, set up objects necessary to compute QoIs

In [ ]:
COMM = dl.MPI.comm_world
STUDY_DIR = "/storage1/transfer/gtp/sub-00101/mesh_refinement_study"
MESH_LEVELS = [0, 1, 2, 3] # 0 is the coarsest mesh

# report mesh info
BASE_RESOLUTION = 32
num_vertices_32 = []

for level in MESH_LEVELS:
    print(f"Mesh level: {level}")
    MESH_FPATH = os.path.join(STUDY_DIR, f"mesh_{BASE_RESOLUTION}_refine_{level}", f"full{BASE_RESOLUTION}-all.h5")
    mesh = load_mesh(COMM, MESH_FPATH)
    num_vertices_32.append(mesh.num_vertices())
    # report_mesh_info(mesh)

num_vertices_32 = np.array(num_vertices_32)

# read back NIfTI files and compute L2 error at the voxel level
ref_nifti = os.path.join(STUDY_DIR, f"mesh_{BASE_RESOLUTION}_refine_{MESH_LEVELS[-1]}", "observed.nii")
ref_img = nibabel.load(ref_nifti)
ref_data = ref_img.get_fdata()

voxel_l2_errors_32 = []

for level in MESH_LEVELS[:-1]: # skip the finest level since it's the reference
    obs_nifti = os.path.join(STUDY_DIR, f"mesh_{BASE_RESOLUTION}_refine_{level}", "observed.nii")
    obs_img = nibabel.load(obs_nifti)
    obs_data = obs_img.get_fdata()

    # compute L2 error at the voxel level
    error = np.sqrt(np.sum((obs_data - ref_data)**2))
    voxel_l2_errors_32.append(error)

In [ ]:
COMM = dl.MPI.comm_world
STUDY_DIR = "/storage1/transfer/gtp/sub-00101/mesh_refinement_study"
MESH_LEVELS = [0, 1, 2] # 0 is the coarsest mesh

# report mesh info
BASE_RESOLUTION = 64
num_vertices_64 = []

for level in MESH_LEVELS:
    print(f"Mesh level: {level}")
    MESH_FPATH = os.path.join(STUDY_DIR, f"mesh_{BASE_RESOLUTION}_refine_{level}", f"full{BASE_RESOLUTION}-all.h5")
    mesh = load_mesh(COMM, MESH_FPATH)
    num_vertices_64.append(mesh.num_vertices())
    # report_mesh_info(mesh)

num_vertices_64 = np.array(num_vertices_64)

# read back NIfTI files and compute L2 error at the voxel level
ref_nifti = os.path.join(STUDY_DIR, f"mesh_{BASE_RESOLUTION}_refine_{MESH_LEVELS[-1]}", "observed.nii")
ref_img = nibabel.load(ref_nifti)
ref_data = ref_img.get_fdata()

voxel_l2_errors_64 = []

for level in MESH_LEVELS[:-1]: # skip the finest level since it's the reference
    obs_nifti = os.path.join(STUDY_DIR, f"mesh_{BASE_RESOLUTION}_refine_{level}", "observed.nii")
    obs_img = nibabel.load(obs_nifti)
    obs_data = obs_img.get_fdata()

    # compute L2 error at the voxel level
    error = np.sqrt(np.sum((obs_data - ref_data)**2))
    voxel_l2_errors_64.append(error)

In [ ]:
support = os.path.join(STUDY_DIR, f"mesh_{BASE_RESOLUTION}_refine_{MESH_LEVELS[-1]}", "domain_support.nii")
nnz = np.count_nonzero(nibabel.load(support).get_fdata())
stddev = np.sqrt(nnz) * 0.02

print(f"Expected standard error: {stddev:.2f}")

In [ ]:
num_vertices_64

In [ ]:
plt.figure(figsize=(8, 6))
fig, ax = plt.subplots()

# plot N^(-1/2) reference line
theory = np.power(num_vertices_32[:-1], -2 / 3) * voxel_l2_errors_32[0] / np.power(num_vertices_32[0], -2 / 3)
plt.loglog(num_vertices_32[:-1], 0.5 * theory, linestyle="--", color=gray, alpha=0.8)
ax.text(1.1 * num_vertices_32[-2], 0.4 * theory[-1], r"$\mathcal{O}(N^{-2/3})$", fontsize=14, color=gray)

# plot the noise level
ax.axhline(y=stddev, linestyle="-.", color=red)
ax.text(0.9 * num_vertices_32[-2], 1.1 * stddev, "Noise level", fontsize=14, color=red, ha="left", alpha=0.8)

ax.loglog(num_vertices_32[:-1], voxel_l2_errors_32, "o-", color=utorange, markersize=6, linewidth=1)
ax.loglog(num_vertices_64[:-1], voxel_l2_errors_64, "o--", color=utblue, markersize=6, linewidth=1)

# plot a single black star for the mesh that is used
ax.loglog(num_vertices_64[0], voxel_l2_errors_64[0], "*", color=utblue, markersize=12, label="Mesh used for experiments")

plt.xticks([1e4, 1e5, 1e6])
plt.xlim(1.5e4, 3e6)
plt.ylim(1, 80)

# if BASE_RESOLUTION == 32:
#     plt.xticks([1e4, 1e5, 1e6])
#     plt.xlim(1.5e4, 3e6)
#     plt.ylim(4, 80)
# elif BASE_RESOLUTION == 64:
#     plt.xticks([1e5, 1e6])
#     plt.xlim(9e4, 3e6)
#     plt.ylim(2, 40)

plt.xlabel(r"Number of Vertices ($N$)", fontsize=16, fontweight="bold")
plt.ylabel(r"Error $\| \mathcal{B}(u^{(i)}_{h}) - \mathcal{B}(u^{\dagger}_{h})) \|_2$", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig("mesh_refinement_convergence.pdf", dpi=300)

### Eigenvalue Analysis

In [ ]:
SUBDIR = "/storage1/transfer/gtp/sub-00101"

evals_0 = np.loadtxt(os.path.join(SUBDIR, "rdtx_freq14_bip", "rdtx_freq14_eigenvalues.txt"))
evals_1 = np.loadtxt(os.path.join(SUBDIR, "ref_rdtx_freq14_bip", "rdtx_freq14_eigenvalues.txt"))

evals_0 = evals_0[:50]
evals_1 = evals_1[:50]

fig, ax = plt.subplots(figsize=(6, 3))

numbers = np.arange(len(evals_0))

plt.semilogy(numbers, evals_0, "x", color=utorange, label="Base Mesh", lw=1.5, ms=6)
plt.semilogy(numbers, evals_1, "o", color=utblue, label="Uniformly Refined", lw=1.5, ms=6, markerfacecolor="none")

# labels, ticks, etc
plt.xlabel('Number', fontsize=12)
plt.ylabel(r'Eigenvalue $\lambda_i$', fontsize=12)
plt.xticks([0, 10, 20, 30, 40, 50])
# plt.xlim([-1, 72])
plt.legend()
plt.savefig('mesh_independent_spectrum.pdf', bbox_inches='tight', dpi=300)
plt.show()